# ID-MAS: Instructional Design Multi-Agent System

LangGraph 기반 Iterative Scaffolding Pipeline을 활용한 Knowledge Distillation 프레임워크.

**전체 파이프라인:**
1. **ID-MAS Training** - Teacher 모델이 Student 모델을 반복적으로 가르치며 SFT 데이터 생성
2. **LlamaFactory SFT** - 생성된 데이터로 LoRA Fine-tuning 수행
3. **Evaluation** - Baseline vs ID-MAS SFT 모델 성능 비교

**실험 구성:**

| ID | Student Model | Teacher Model | Type |
|----|--------------|---------------|------|
| [1]~[6] | Qwen3-0.6B ~ 32B | Same as Student | Self-Distillation |
| [7]~[8] | Qwen3-4B, 8B | GPT-5.2 | Cross-Model Distillation |

**평가 도메인 & 데이터셋:**

| Domain | Training | Evaluation |
|--------|----------|------------|
| Math | GSM8K, MATH | GSM8K, MATH, SVAMP, ASDiv, MAWPS |
| Logical | ReClor | ReClor, ANLI-R2, ANLI-R3, BBH |
| Commonsense | ARC-C | ARC-C, StrategyQA, OpenBookQA |

## 0. Environment Setup

Google Drive 마운트, 프로젝트 경로 설정, 의존성 설치를 수행합니다.

> **`BASE_DIR`을 본인의 프로젝트 경로로 수정하세요.**

In [ ]:
# from google.colab import drive
# drive.mount('/content/gdrive/')

In [ ]:
import os

# ============================================================
# === 여기서 프로젝트 루트 경로를 설정하세요 ===
BASE_DIR = "/root/to/ID-MAS/ID-MAS"
# ============================================================

IDMAS_ROOT = BASE_DIR
LF_ROOT = os.path.join(BASE_DIR, "LlamaFactory")

os.chdir(IDMAS_ROOT)
print(f"Working directory: {os.getcwd()}")
print(f"ID-MAS root: {IDMAS_ROOT}")
print(f"LlamaFactory root: {LF_ROOT}")

In [ ]:
# !pip install -r requirements.txt

## 1. ID-MAS Training - Iterative Scaffolding Pipeline

`main.py --mode train`을 실행하여 Teacher-Student 반복 학습을 수행합니다.

**파이프라인 단계:**
1. **Instructional Design** - Teacher가 도메인별 학습 목표(Performance Objectives) 설계
2. **Learning Loop** (최대 5회 반복) - Student 응답 → Teacher 평가 → 소크라테스식 피드백
3. **SFT Data Generation** - 학습 결과를 3가지 Case로 분류하여 SFT 데이터 생성
   - **Case A**: Independent Performance Mastery (PO 충족)
   - **Case B**: Scaffolded & Coached Mastery (스캐폴딩 후 PO 충족)
   - **Case C**: Teacher Modeling Distillation (교사 시범)

### 1.1 Self-Distillation (Teacher = Student)

동일 모델을 Teacher와 Student로 사용하는 자기증류 실험입니다.
각 모델이 자기 자신을 가르치며 SFT 데이터를 생성합니다.

In [ ]:
## [1] Student Model: Qwen3-0.6B / Teacher Model: Qwen3-0.6B
!python main.py --mode train --domain math --train-dataset gsm8k \
    --student-model "Qwen/Qwen3-0.6B" \
    --teacher-model "Qwen/Qwen3-0.6B" \
    --student-gpu 0

!python main.py --mode train --domain math --train-dataset math \
    --student-model "Qwen/Qwen3-0.6B" \
    --teacher-model "Qwen/Qwen3-0.6B" \
    --student-gpu 0

!python main.py --mode train --domain logical --train-dataset reclor \
    --student-model "Qwen/Qwen3-0.6B" \
    --teacher-model "Qwen/Qwen3-0.6B" \
    --student-gpu 0

!python main.py --mode train --domain commonsense --train-dataset arc_c \
    --student-model "Qwen/Qwen3-0.6B" \
    --teacher-model "Qwen/Qwen3-0.6B" \
    --student-gpu 0

In [ ]:
## [2] Student Model: Qwen3-1.7B / Teacher Model: Qwen3-1.7B
!python main.py --mode train --domain math --train-dataset gsm8k \
    --student-model "Qwen/Qwen3-1.7B" \
    --teacher-model "Qwen/Qwen3-1.7B" \
    --student-gpu 0

!python main.py --mode train --domain math --train-dataset math \
    --student-model "Qwen/Qwen3-1.7B" \
    --teacher-model "Qwen/Qwen3-1.7B" \
    --student-gpu 0

!python main.py --mode train --domain logical --train-dataset reclor \
    --student-model "Qwen/Qwen3-1.7B" \
    --teacher-model "Qwen/Qwen3-1.7B" \
    --student-gpu 0

!python main.py --mode train --domain commonsense --train-dataset arc_c \
    --student-model "Qwen/Qwen3-1.7B" \
    --teacher-model "Qwen/Qwen3-1.7B" \
    --student-gpu 0

In [ ]:
## [3] Student Model: Qwen3-4B / Teacher Model: Qwen3-4B
!python main.py --mode train --domain math --train-dataset gsm8k \
    --student-model "Qwen/Qwen3-4B" \
    --teacher-model "Qwen/Qwen3-4B" \
    --student-gpu 0

!python main.py --mode train --domain math --train-dataset math \
    --student-model "Qwen/Qwen3-4B" \
    --teacher-model "Qwen/Qwen3-4B" \
    --student-gpu 0

!python main.py --mode train --domain logical --train-dataset reclor \
    --student-model "Qwen/Qwen3-4B" \
    --teacher-model "Qwen/Qwen3-4B" \
    --student-gpu 0

!python main.py --mode train --domain commonsense --train-dataset arc_c \
    --student-model "Qwen/Qwen3-4B" \
    --teacher-model "Qwen/Qwen3-4B" \
    --student-gpu 0

In [ ]:
## [4] Student Model: Qwen3-8B / Teacher Model: Qwen3-8B
!python main.py --mode train --domain math --train-dataset gsm8k \
    --student-model "Qwen/Qwen3-8B" \
    --teacher-model "Qwen/Qwen3-8B" \
    --student-gpu 0

!python main.py --mode train --domain math --train-dataset math \
    --student-model "Qwen/Qwen3-8B" \
    --teacher-model "Qwen/Qwen3-8B" \
    --student-gpu 0

!python main.py --mode train --domain logical --train-dataset reclor \
    --student-model "Qwen/Qwen3-8B" \
    --teacher-model "Qwen/Qwen3-8B" \
    --student-gpu 0

!python main.py --mode train --domain commonsense --train-dataset arc_c \
    --student-model "Qwen/Qwen3-8B" \
    --teacher-model "Qwen/Qwen3-8B" \
    --student-gpu 0

In [ ]:
## [5] Student Model: Qwen3-14B / Teacher Model: Qwen3-14B
!python main.py --mode train --domain math --train-dataset gsm8k \
    --student-model "Qwen/Qwen3-14B" \
    --teacher-model "Qwen/Qwen3-14B" \
    --student-gpu 0

!python main.py --mode train --domain math --train-dataset math \
    --student-model "Qwen/Qwen3-14B" \
    --teacher-model "Qwen/Qwen3-14B" \
    --student-gpu 0

!python main.py --mode train --domain logical --train-dataset reclor \
    --student-model "Qwen/Qwen3-14B" \
    --teacher-model "Qwen/Qwen3-14B" \
    --student-gpu 0

!python main.py --mode train --domain commonsense --train-dataset arc_c \
    --student-model "Qwen/Qwen3-14B" \
    --teacher-model "Qwen/Qwen3-14B" \
    --student-gpu 0

In [ ]:
## [6] Student Model: Qwen3-32B / Teacher Model: Qwen3-32B
!python main.py --mode train --domain math --train-dataset gsm8k \
    --student-model "Qwen/Qwen3-32B" \
    --teacher-model "Qwen/Qwen3-32B" \
    --student-gpu 0

!python main.py --mode train --domain math --train-dataset math \
    --student-model "Qwen/Qwen3-32B" \
    --teacher-model "Qwen/Qwen3-32B" \
    --student-gpu 0

!python main.py --mode train --domain logical --train-dataset reclor \
    --student-model "Qwen/Qwen3-32B" \
    --teacher-model "Qwen/Qwen3-32B" \
    --student-gpu 0

!python main.py --mode train --domain commonsense --train-dataset arc_c \
    --student-model "Qwen/Qwen3-32B" \
    --teacher-model "Qwen/Qwen3-32B" \
    --student-gpu 0

### 1.2 Cross-Model Distillation (Teacher = GPT-5.2)

GPT-5.2를 Teacher로 사용하여 더 작은 Student 모델(Qwen3-4B, 8B)을 학습시킵니다.
대형 모델의 추론 능력을 소형 모델로 전이하는 Cross-Model Distillation 실험입니다.

In [ ]:
## [7] Student Model: Qwen3-4B / Teacher Model: gpt-5.2
!python main.py --mode train --domain math --train-dataset gsm8k \
    --student-model "Qwen/Qwen3-4B" \
    --teacher-model "gpt-5.2" \
    --student-gpu 0

!python main.py --mode train --domain math --train-dataset math \
    --student-model "Qwen/Qwen3-4B" \
    --teacher-model "gpt-5.2" \
    --student-gpu 0

!python main.py --mode train --domain logical --train-dataset reclor \
    --student-model "Qwen/Qwen3-4B" \
    --teacher-model "gpt-5.2" \
    --student-gpu 0

!python main.py --mode train --domain commonsense --train-dataset arc_c \
    --student-model "Qwen/Qwen3-4B" \
    --teacher-model "gpt-5.2" \
    --student-gpu 0

In [ ]:
## [8] Student Model: Qwen3-8B / Teacher Model: gpt-5.2
!python main.py --mode train --domain math --train-dataset gsm8k \
    --student-model "Qwen/Qwen3-8B" \
    --teacher-model "gpt-5.2" \
    --student-gpu 0

!python main.py --mode train --domain math --train-dataset math \
    --student-model "Qwen/Qwen3-8B" \
    --teacher-model "gpt-5.2" \
    --student-gpu 0

!python main.py --mode train --domain logical --train-dataset reclor \
    --student-model "Qwen/Qwen3-8B" \
    --teacher-model "gpt-5.2" \
    --student-gpu 0

!python main.py --mode train --domain commonsense --train-dataset arc_c \
    --student-model "Qwen/Qwen3-8B" \
    --teacher-model "gpt-5.2" \
    --student-gpu 0

## 2. LlamaFactory - LoRA SFT Fine-tuning

ID-MAS에서 생성한 SFT 데이터를 사용하여 [LlamaFactory](https://github.com/hiyouga/LlamaFactory)로 LoRA Fine-tuning을 수행합니다.

**SFT 설정:**
- **Method**: LoRA (rank=8, target=all)
- **Template**: Qwen3
- **Training**: 3 epochs, cosine LR scheduler, warmup 0.1
- **Precision**: bf16

### 2.0 LlamaFactory Setup

LlamaFactory 설치 및 SFT 데이터/YAML 설정 파일을 자동 생성합니다.

In [ ]:
# !git clone https://github.com/hiyouga/LlamaFactory.git

In [ ]:
# os.chdir(LF_ROOT)
# !pip install -e ".[torch,metrics]"

In [ ]:
import json
import shutil
from pathlib import Path
from dotenv import load_dotenv

# === Paths (BASE_DIR 기반) ===
IDMAS_PATH = Path(IDMAS_ROOT)
LF_PATH = Path(LF_ROOT)
LF_DATA_DIR = LF_PATH / "data"
LF_EXAMPLES_DIR = LF_PATH / "examples" / "train_custom"
OUTPUTS_DIR = IDMAS_PATH / "outputs"

# === Load .env ===
load_dotenv(IDMAS_PATH / ".env")
HF_TOKEN = os.environ["HF_TOKEN"]

# === Experiment Definitions ===
EXPERIMENTS = [
    {"id": 1, "student": "Qwen/Qwen3-0.6B", "short": "Qwen3-0.6B", "teacher_short": "Qwen3-0.6B", "hf": "qwen3-0.6b", "is_gpt": False},
    {"id": 2, "student": "Qwen/Qwen3-1.7B", "short": "Qwen3-1.7B", "teacher_short": "Qwen3-1.7B", "hf": "qwen3-1.7b", "is_gpt": False},
    {"id": 3, "student": "Qwen/Qwen3-4B",   "short": "Qwen3-4B",   "teacher_short": "Qwen3-4B",   "hf": "qwen3-4b",   "is_gpt": False},
    {"id": 4, "student": "Qwen/Qwen3-8B",   "short": "Qwen3-8B",   "teacher_short": "Qwen3-8B",   "hf": "qwen3-8b",   "is_gpt": False},
    {"id": 5, "student": "Qwen/Qwen3-14B",  "short": "Qwen3-14B",  "teacher_short": "Qwen3-14B",  "hf": "qwen3-14b",  "is_gpt": False},
    {"id": 6, "student": "Qwen/Qwen3-32B",  "short": "Qwen3-32B",  "teacher_short": "Qwen3-32B",  "hf": "qwen3-32b",  "is_gpt": False},
    {"id": 7, "student": "Qwen/Qwen3-4B",   "short": "Qwen3-4B",   "teacher_short": "gpt-5.2",    "hf": "qwen3-4b",   "is_gpt": True},
    {"id": 8, "student": "Qwen/Qwen3-8B",   "short": "Qwen3-8B",   "teacher_short": "gpt-5.2",    "hf": "qwen3-8b",   "is_gpt": True},
]

DOMAIN_DATASETS = {"math": ["gsm8k", "math"], "logical": ["reclor"], "commonsense": ["arc_c"]}

HYPERPARAMS = {
    "0.6B": {"batch": 8,  "grad_accum": 2,  "ds": None},
    "1.7B": {"batch": 4,  "grad_accum": 4,  "ds": None},
    "4B":   {"batch": 2,  "grad_accum": 8,  "ds": None},
    "8B":   {"batch": 1,  "grad_accum": 16, "ds": None},
    "14B":  {"batch": 1,  "grad_accum": 16, "ds": None},
    "32B":  {"batch": 1,  "grad_accum": 16, "ds": "examples/deepspeed/ds_z3_config.json"},
}

# === Load dataset_info.json ===
dataset_info_path = LF_DATA_DIR / "dataset_info.json"
with open(dataset_info_path, 'r', encoding='utf-8') as f:
    dataset_info = json.load(f)

print("=== Data Preparation & YAML Generation ===\n")
count = 0

for exp in EXPERIMENTS:
    size = exp["short"].split("-")[-1]
    hp = HYPERPARAMS[size]
    suffix = "_gpt52" if exp["is_gpt"] else ""
    hf_suffix = "-gpt52" if exp["is_gpt"] else ""

    for domain, datasets in DOMAIN_DATASETS.items():
        for dataset in datasets:
            # 1. Copy SFT data
            src = OUTPUTS_DIR / domain / "train" / exp["teacher_short"] / exp["short"] / f"{dataset}_train_id-mas_{exp['short']}.json"
            lf_name = f"idmas_{domain}_{dataset}_{exp['short']}{suffix}"
            dst = LF_DATA_DIR / f"{lf_name}.json"

            if src.exists():
                shutil.copy2(src, dst)
                print(f"[Copy] {src.name} -> {dst.name}")
            else:
                print(f"[WARN] Not found: {src}")
                continue

            # 2. Register in dataset_info.json
            dataset_info[lf_name] = {"file_name": f"{lf_name}.json"}

            # 3. Generate train.yaml
            dir_suffix = f"{domain}_{dataset}{suffix}"
            yaml_dir = LF_EXAMPLES_DIR / exp["short"] / dir_suffix
            yaml_dir.mkdir(parents=True, exist_ok=True)

            ds_line = f"\ndeepspeed: {hp['ds']}" if hp["ds"] else ""
            train_yaml = f"""### model
model_name_or_path: {exp["student"]}
trust_remote_code: true

### method
stage: sft
do_train: true
finetuning_type: lora
lora_rank: 8
lora_target: all

### dataset
dataset: {lf_name}
template: qwen3_nothink
cutoff_len: 2048

### output
output_dir: saves/{exp["short"]}/lora/{dir_suffix}
logging_steps: 10
save_steps: 500
plot_loss: true
overwrite_output_dir: true

### train
per_device_train_batch_size: {hp["batch"]}
gradient_accumulation_steps: {hp["grad_accum"]}
learning_rate: 1.0e-4
num_train_epochs: 3.0
lr_scheduler_type: cosine
warmup_ratio: 0.1
bf16: true{ds_line}
"""
            with open(yaml_dir / "train.yaml", 'w') as f:
                f.write(train_yaml)

            # 4. Generate merge.yaml (with HuggingFace upload)
            repo_id = f"SaFD-00/{exp['hf']}-id-mas-{domain}-{dataset}{hf_suffix}"
            merge_yaml = f"""model_name_or_path: {exp["student"]}
adapter_name_or_path: saves/{exp["short"]}/lora/{dir_suffix}
template: qwen3_nothink
finetuning_type: lora
trust_remote_code: true
export_dir: saves/{exp["short"]}/merged/{dir_suffix}
export_size: 5
export_device: cpu
export_legacy_format: false
hf_hub_token: {HF_TOKEN}
export_hub_model_id: {repo_id}
"""
            with open(yaml_dir / "merge.yaml", 'w') as f:
                f.write(merge_yaml)

            count += 1
            print(f"[YAML] {exp['short']}/{dir_suffix}/train.yaml, merge.yaml -> {repo_id}")

# 5. Save dataset_info.json
with open(dataset_info_path, 'w', encoding='utf-8') as f:
    json.dump(dataset_info, f, ensure_ascii=False, indent=2)

print(f"\n=== Done: {count} datasets copied, registered, YAML generated ===")

### 2.1 Self-Distillation SFT Training

Self-Distillation 실험 [1]~[6]의 LoRA SFT 학습을 수행합니다.
각 실험은 4개 도메인-데이터셋 조합(math/gsm8k, math/math, logical/reclor, commonsense/arc_c)을 학습합니다.

| Model | Batch Size | Grad Accum | Effective Batch |
|-------|-----------|------------|----------------|
| Qwen3-0.6B | 8 | 2 | 16 |
| Qwen3-1.7B | 4 | 4 | 16 |
| Qwen3-4B | 2 | 8 | 16 |
| Qwen3-8B~32B | 1 | 16 | 16 |

In [ ]:
os.chdir(LF_ROOT)

## [1] Qwen3-0.6B LoRA SFT Training (Teacher=Qwen3-0.6B)

# math - gsm8k
!llamafactory-cli train examples/train_custom/Qwen3-0.6B/math_gsm8k/train.yaml

# math - math
!llamafactory-cli train examples/train_custom/Qwen3-0.6B/math_math/train.yaml

# logical - reclor
!llamafactory-cli train examples/train_custom/Qwen3-0.6B/logical_reclor/train.yaml

# commonsense - arc_c
!llamafactory-cli train examples/train_custom/Qwen3-0.6B/commonsense_arc_c/train.yaml

In [ ]:
os.chdir(LF_ROOT)

## [2] Qwen3-1.7B LoRA SFT Training (Teacher=Qwen3-1.7B)

# math - gsm8k
!llamafactory-cli train examples/train_custom/Qwen3-1.7B/math_gsm8k/train.yaml

# math - math
!llamafactory-cli train examples/train_custom/Qwen3-1.7B/math_math/train.yaml

# logical - reclor
!llamafactory-cli train examples/train_custom/Qwen3-1.7B/logical_reclor/train.yaml

# commonsense - arc_c
!llamafactory-cli train examples/train_custom/Qwen3-1.7B/commonsense_arc_c/train.yaml

In [ ]:
os.chdir(LF_ROOT)

## [3] Qwen3-4B LoRA SFT Training (Teacher=Qwen3-4B)

# math - gsm8k
!llamafactory-cli train examples/train_custom/Qwen3-4B/math_gsm8k/train.yaml

# math - math
!llamafactory-cli train examples/train_custom/Qwen3-4B/math_math/train.yaml

# logical - reclor
!llamafactory-cli train examples/train_custom/Qwen3-4B/logical_reclor/train.yaml

# commonsense - arc_c
!llamafactory-cli train examples/train_custom/Qwen3-4B/commonsense_arc_c/train.yaml

In [ ]:
os.chdir(LF_ROOT)

## [4] Qwen3-8B LoRA SFT Training (Teacher=Qwen3-8B)

# math - gsm8k
!llamafactory-cli train examples/train_custom/Qwen3-8B/math_gsm8k/train.yaml

# math - math
!llamafactory-cli train examples/train_custom/Qwen3-8B/math_math/train.yaml

# logical - reclor
!llamafactory-cli train examples/train_custom/Qwen3-8B/logical_reclor/train.yaml

# commonsense - arc_c
!llamafactory-cli train examples/train_custom/Qwen3-8B/commonsense_arc_c/train.yaml

In [ ]:
os.chdir(LF_ROOT)

## [6] Qwen3-32B LoRA SFT Training (Teacher=Qwen3-32B)

# math - gsm8k
!llamafactory-cli train examples/train_custom/Qwen3-32B/math_gsm8k/train.yaml

# math - math
!llamafactory-cli train examples/train_custom/Qwen3-32B/math_math/train.yaml

# logical - reclor
!llamafactory-cli train examples/train_custom/Qwen3-32B/logical_reclor/train.yaml

# commonsense - arc_c
!llamafactory-cli train examples/train_custom/Qwen3-32B/commonsense_arc_c/train.yaml

### 2.2 Cross-Model Distillation SFT Training

GPT-5.2 Teacher로 생성한 SFT 데이터를 사용하여 Qwen3-4B, 8B를 Fine-tuning합니다.

In [ ]:
os.chdir(LF_ROOT)

## [7] Qwen3-4B LoRA SFT Training (Teacher=gpt-5.2)

# math - gsm8k
!llamafactory-cli train examples/train_custom/Qwen3-4B/math_gsm8k_gpt52/train.yaml

# math - math
!llamafactory-cli train examples/train_custom/Qwen3-4B/math_math_gpt52/train.yaml

# logical - reclor
!llamafactory-cli train examples/train_custom/Qwen3-4B/logical_reclor_gpt52/train.yaml

# commonsense - arc_c
!llamafactory-cli train examples/train_custom/Qwen3-4B/commonsense_arc_c_gpt52/train.yaml

In [ ]:
os.chdir(LF_ROOT)

## [8] Qwen3-8B LoRA SFT Training (Teacher=gpt-5.2)

# math - gsm8k
!llamafactory-cli train examples/train_custom/Qwen3-8B/math_gsm8k_gpt52/train.yaml

# math - math
!llamafactory-cli train examples/train_custom/Qwen3-8B/math_math_gpt52/train.yaml

# logical - reclor
!llamafactory-cli train examples/train_custom/Qwen3-8B/logical_reclor_gpt52/train.yaml

# commonsense - arc_c
!llamafactory-cli train examples/train_custom/Qwen3-8B/commonsense_arc_c_gpt52/train.yaml

### 2.3 Merge & Upload to HuggingFace

LoRA 어댑터를 기본 모델에 병합(merge)하고 HuggingFace Hub에 업로드합니다.
병합된 모델은 `SaFD-00/{model}-{domain}-{dataset}` 형식으로 업로드됩니다.

In [ ]:
os.chdir(LF_ROOT)

## [1] Qwen3-0.6B LoRA Merge (Teacher=Qwen3-0.6B)

# math - gsm8k
!llamafactory-cli export examples/train_custom/Qwen3-0.6B/math_gsm8k/merge.yaml

# math - math
!llamafactory-cli export examples/train_custom/Qwen3-0.6B/math_math/merge.yaml

# logical - reclor
!llamafactory-cli export examples/train_custom/Qwen3-0.6B/logical_reclor/merge.yaml

# commonsense - arc_c
!llamafactory-cli export examples/train_custom/Qwen3-0.6B/commonsense_arc_c/merge.yaml

In [ ]:
os.chdir(LF_ROOT)

## [2] Qwen3-1.7B LoRA Merge (Teacher=Qwen3-1.7B)

# math - gsm8k
!llamafactory-cli export examples/train_custom/Qwen3-1.7B/math_gsm8k/merge.yaml

# math - math
!llamafactory-cli export examples/train_custom/Qwen3-1.7B/math_math/merge.yaml

# logical - reclor
!llamafactory-cli export examples/train_custom/Qwen3-1.7B/logical_reclor/merge.yaml

# commonsense - arc_c
!llamafactory-cli export examples/train_custom/Qwen3-1.7B/commonsense_arc_c/merge.yaml

In [ ]:
os.chdir(LF_ROOT)

## [3] Qwen3-4B LoRA Merge (Teacher=Qwen3-4B)

# math - gsm8k
!llamafactory-cli export examples/train_custom/Qwen3-4B/math_gsm8k/merge.yaml

# math - math
!llamafactory-cli export examples/train_custom/Qwen3-4B/math_math/merge.yaml

# logical - reclor
!llamafactory-cli export examples/train_custom/Qwen3-4B/logical_reclor/merge.yaml

# commonsense - arc_c
!llamafactory-cli export examples/train_custom/Qwen3-4B/commonsense_arc_c/merge.yaml

In [ ]:
os.chdir(LF_ROOT)

## [4] Qwen3-8B LoRA Merge (Teacher=Qwen3-8B)

# math - gsm8k
!llamafactory-cli export examples/train_custom/Qwen3-8B/math_gsm8k/merge.yaml

# math - math
!llamafactory-cli export examples/train_custom/Qwen3-8B/math_math/merge.yaml

# logical - reclor
!llamafactory-cli export examples/train_custom/Qwen3-8B/logical_reclor/merge.yaml

# commonsense - arc_c
!llamafactory-cli export examples/train_custom/Qwen3-8B/commonsense_arc_c/merge.yaml

In [ ]:
os.chdir(LF_ROOT)

## [5] Qwen3-14B LoRA Merge (Teacher=Qwen3-14B)

# math - gsm8k
!llamafactory-cli export examples/train_custom/Qwen3-14B/math_gsm8k/merge.yaml

# math - math
!llamafactory-cli export examples/train_custom/Qwen3-14B/math_math/merge.yaml

# logical - reclor
!llamafactory-cli export examples/train_custom/Qwen3-14B/logical_reclor/merge.yaml

# commonsense - arc_c
!llamafactory-cli export examples/train_custom/Qwen3-14B/commonsense_arc_c/merge.yaml

In [ ]:
os.chdir(LF_ROOT)

## [6] Qwen3-32B LoRA Merge (Teacher=Qwen3-32B)

# math - gsm8k
!llamafactory-cli export examples/train_custom/Qwen3-32B/math_gsm8k/merge.yaml

# math - math
!llamafactory-cli export examples/train_custom/Qwen3-32B/math_math/merge.yaml

# logical - reclor
!llamafactory-cli export examples/train_custom/Qwen3-32B/logical_reclor/merge.yaml

# commonsense - arc_c
!llamafactory-cli export examples/train_custom/Qwen3-32B/commonsense_arc_c/merge.yaml

In [ ]:
os.chdir(LF_ROOT)

## [7] Qwen3-4B LoRA Merge (Teacher=gpt-5.2)

# math - gsm8k
!llamafactory-cli export examples/train_custom/Qwen3-4B/math_gsm8k_gpt52/merge.yaml

# math - math
!llamafactory-cli export examples/train_custom/Qwen3-4B/math_math_gpt52/merge.yaml

# logical - reclor
!llamafactory-cli export examples/train_custom/Qwen3-4B/logical_reclor_gpt52/merge.yaml

# commonsense - arc_c
!llamafactory-cli export examples/train_custom/Qwen3-4B/commonsense_arc_c_gpt52/merge.yaml

In [ ]:
os.chdir(LF_ROOT)

## [8] Qwen3-8B LoRA Merge (Teacher=gpt-5.2)

# math - gsm8k
!llamafactory-cli export examples/train_custom/Qwen3-8B/math_gsm8k_gpt52/merge.yaml

# math - math
!llamafactory-cli export examples/train_custom/Qwen3-8B/math_math_gpt52/merge.yaml

# logical - reclor
!llamafactory-cli export examples/train_custom/Qwen3-8B/logical_reclor_gpt52/merge.yaml

# commonsense - arc_c
!llamafactory-cli export examples/train_custom/Qwen3-8B/commonsense_arc_c_gpt52/merge.yaml

## 3. Evaluation

Baseline(원본 모델) 대비 ID-MAS SFT 모델의 성능을 비교 평가합니다.

**평가 방식:**
- 각 모델에 대해 **Baseline** (Qwen/Qwen3-*B 원본) 평가를 먼저 수행
- 이후 **SFT 모델** (SaFD-00/*) 평가를 도메인별로 수행
- In-domain (학습 데이터셋) + Out-of-domain (미학습 데이터셋) 모두 평가

### 3.1 Self-Distillation Evaluation

실험 [1]~[6]: 동일 모델 자기증류 후 성능 평가

In [ ]:
os.chdir(IDMAS_ROOT)

## [1] Qwen3-0.6B Evaluation (Teacher=Qwen3-0.6B)

### Baseline (Qwen/Qwen3-0.6B)
!python main.py --mode eval --domain math --eval-dataset gsm8k --student-model Qwen/Qwen3-0.6B --student-gpu 0
!python main.py --mode eval --domain math --eval-dataset math --student-model Qwen/Qwen3-0.6B --student-gpu 0
!python main.py --mode eval --domain math --eval-dataset svamp --student-model Qwen/Qwen3-0.6B --student-gpu 0
!python main.py --mode eval --domain math --eval-dataset asdiv --student-model Qwen/Qwen3-0.6B --student-gpu 0
!python main.py --mode eval --domain math --eval-dataset mawps --student-model Qwen/Qwen3-0.6B --student-gpu 0
!python main.py --mode eval --domain logical --eval-dataset reclor --student-model Qwen/Qwen3-0.6B --student-gpu 0
!python main.py --mode eval --domain logical --eval-dataset anli_r2 --student-model Qwen/Qwen3-0.6B --student-gpu 0
!python main.py --mode eval --domain logical --eval-dataset anli_r3 --student-model Qwen/Qwen3-0.6B --student-gpu 0
!python main.py --mode eval --domain logical --eval-dataset bbh --student-model Qwen/Qwen3-0.6B --student-gpu 0
!python main.py --mode eval --domain commonsense --eval-dataset arc_c --student-model Qwen/Qwen3-0.6B --student-gpu 0
!python main.py --mode eval --domain commonsense --eval-dataset strategyqa --student-model Qwen/Qwen3-0.6B --student-gpu 0
!python main.py --mode eval --domain commonsense --eval-dataset openbookqa --student-model Qwen/Qwen3-0.6B --student-gpu 0

### SFT - math/gsm8k (SaFD-00/qwen3-0.6b-id-mas-math-gsm8k)
!python main.py --mode eval --domain math --eval-dataset gsm8k --student-model SaFD-00/qwen3-0.6b-id-mas-math-gsm8k --student-gpu 0
!python main.py --mode eval --domain math --eval-dataset math --student-model SaFD-00/qwen3-0.6b-id-mas-math-gsm8k --student-gpu 0
!python main.py --mode eval --domain math --eval-dataset svamp --student-model SaFD-00/qwen3-0.6b-id-mas-math-gsm8k --student-gpu 0
!python main.py --mode eval --domain math --eval-dataset asdiv --student-model SaFD-00/qwen3-0.6b-id-mas-math-gsm8k --student-gpu 0
!python main.py --mode eval --domain math --eval-dataset mawps --student-model SaFD-00/qwen3-0.6b-id-mas-math-gsm8k --student-gpu 0

### SFT - math/math (SaFD-00/qwen3-0.6b-id-mas-math-math)
!python main.py --mode eval --domain math --eval-dataset gsm8k --student-model SaFD-00/qwen3-0.6b-id-mas-math-math --student-gpu 0
!python main.py --mode eval --domain math --eval-dataset math --student-model SaFD-00/qwen3-0.6b-id-mas-math-math --student-gpu 0
!python main.py --mode eval --domain math --eval-dataset svamp --student-model SaFD-00/qwen3-0.6b-id-mas-math-math --student-gpu 0
!python main.py --mode eval --domain math --eval-dataset asdiv --student-model SaFD-00/qwen3-0.6b-id-mas-math-math --student-gpu 0
!python main.py --mode eval --domain math --eval-dataset mawps --student-model SaFD-00/qwen3-0.6b-id-mas-math-math --student-gpu 0

### SFT - logical/reclor (SaFD-00/qwen3-0.6b-id-mas-logical-reclor)
!python main.py --mode eval --domain logical --eval-dataset reclor --student-model SaFD-00/qwen3-0.6b-id-mas-logical-reclor --student-gpu 0
!python main.py --mode eval --domain logical --eval-dataset anli_r2 --student-model SaFD-00/qwen3-0.6b-id-mas-logical-reclor --student-gpu 0
!python main.py --mode eval --domain logical --eval-dataset anli_r3 --student-model SaFD-00/qwen3-0.6b-id-mas-logical-reclor --student-gpu 0
!python main.py --mode eval --domain logical --eval-dataset bbh --student-model SaFD-00/qwen3-0.6b-id-mas-logical-reclor --student-gpu 0

### SFT - commonsense/arc_c (SaFD-00/qwen3-0.6b-id-mas-commonsense-arc_c)
!python main.py --mode eval --domain commonsense --eval-dataset arc_c --student-model SaFD-00/qwen3-0.6b-id-mas-commonsense-arc_c --student-gpu 0
!python main.py --mode eval --domain commonsense --eval-dataset strategyqa --student-model SaFD-00/qwen3-0.6b-id-mas-commonsense-arc_c --student-gpu 0
!python main.py --mode eval --domain commonsense --eval-dataset openbookqa --student-model SaFD-00/qwen3-0.6b-id-mas-commonsense-arc_c --student-gpu 0

In [ ]:
os.chdir(IDMAS_ROOT)

## [2] Qwen3-1.7B Evaluation (Teacher=Qwen3-1.7B)

### Baseline (Qwen/Qwen3-1.7B)
!python main.py --mode eval --domain math --eval-dataset gsm8k --student-model Qwen/Qwen3-1.7B --student-gpu 0
!python main.py --mode eval --domain math --eval-dataset math --student-model Qwen/Qwen3-1.7B --student-gpu 0
!python main.py --mode eval --domain math --eval-dataset svamp --student-model Qwen/Qwen3-1.7B --student-gpu 0
!python main.py --mode eval --domain math --eval-dataset asdiv --student-model Qwen/Qwen3-1.7B --student-gpu 0
!python main.py --mode eval --domain math --eval-dataset mawps --student-model Qwen/Qwen3-1.7B --student-gpu 0
!python main.py --mode eval --domain logical --eval-dataset reclor --student-model Qwen/Qwen3-1.7B --student-gpu 0
!python main.py --mode eval --domain logical --eval-dataset anli_r2 --student-model Qwen/Qwen3-1.7B --student-gpu 0
!python main.py --mode eval --domain logical --eval-dataset anli_r3 --student-model Qwen/Qwen3-1.7B --student-gpu 0
!python main.py --mode eval --domain logical --eval-dataset bbh --student-model Qwen/Qwen3-1.7B --student-gpu 0
!python main.py --mode eval --domain commonsense --eval-dataset arc_c --student-model Qwen/Qwen3-1.7B --student-gpu 0
!python main.py --mode eval --domain commonsense --eval-dataset strategyqa --student-model Qwen/Qwen3-1.7B --student-gpu 0
!python main.py --mode eval --domain commonsense --eval-dataset openbookqa --student-model Qwen/Qwen3-1.7B --student-gpu 0

### SFT - math/gsm8k (SaFD-00/qwen3-1.7b-id-mas-math-gsm8k)
!python main.py --mode eval --domain math --eval-dataset gsm8k --student-model SaFD-00/qwen3-1.7b-id-mas-math-gsm8k --student-gpu 0
!python main.py --mode eval --domain math --eval-dataset math --student-model SaFD-00/qwen3-1.7b-id-mas-math-gsm8k --student-gpu 0
!python main.py --mode eval --domain math --eval-dataset svamp --student-model SaFD-00/qwen3-1.7b-id-mas-math-gsm8k --student-gpu 0
!python main.py --mode eval --domain math --eval-dataset asdiv --student-model SaFD-00/qwen3-1.7b-id-mas-math-gsm8k --student-gpu 0
!python main.py --mode eval --domain math --eval-dataset mawps --student-model SaFD-00/qwen3-1.7b-id-mas-math-gsm8k --student-gpu 0

### SFT - math/math (SaFD-00/qwen3-1.7b-id-mas-math-math)
!python main.py --mode eval --domain math --eval-dataset gsm8k --student-model SaFD-00/qwen3-1.7b-id-mas-math-math --student-gpu 0
!python main.py --mode eval --domain math --eval-dataset math --student-model SaFD-00/qwen3-1.7b-id-mas-math-math --student-gpu 0
!python main.py --mode eval --domain math --eval-dataset svamp --student-model SaFD-00/qwen3-1.7b-id-mas-math-math --student-gpu 0
!python main.py --mode eval --domain math --eval-dataset asdiv --student-model SaFD-00/qwen3-1.7b-id-mas-math-math --student-gpu 0
!python main.py --mode eval --domain math --eval-dataset mawps --student-model SaFD-00/qwen3-1.7b-id-mas-math-math --student-gpu 0

### SFT - logical/reclor (SaFD-00/qwen3-1.7b-id-mas-logical-reclor)
!python main.py --mode eval --domain logical --eval-dataset reclor --student-model SaFD-00/qwen3-1.7b-id-mas-logical-reclor --student-gpu 0
!python main.py --mode eval --domain logical --eval-dataset anli_r2 --student-model SaFD-00/qwen3-1.7b-id-mas-logical-reclor --student-gpu 0
!python main.py --mode eval --domain logical --eval-dataset anli_r3 --student-model SaFD-00/qwen3-1.7b-id-mas-logical-reclor --student-gpu 0
!python main.py --mode eval --domain logical --eval-dataset bbh --student-model SaFD-00/qwen3-1.7b-id-mas-logical-reclor --student-gpu 0

### SFT - commonsense/arc_c (SaFD-00/qwen3-1.7b-id-mas-commonsense-arc_c)
!python main.py --mode eval --domain commonsense --eval-dataset arc_c --student-model SaFD-00/qwen3-1.7b-id-mas-commonsense-arc_c --student-gpu 0
!python main.py --mode eval --domain commonsense --eval-dataset strategyqa --student-model SaFD-00/qwen3-1.7b-id-mas-commonsense-arc_c --student-gpu 0
!python main.py --mode eval --domain commonsense --eval-dataset openbookqa --student-model SaFD-00/qwen3-1.7b-id-mas-commonsense-arc_c --student-gpu 0

In [ ]:
os.chdir(IDMAS_ROOT)

## [3] Qwen3-4B Evaluation (Teacher=Qwen3-4B)

### Baseline (Qwen/Qwen3-4B)
!python main.py --mode eval --domain math --eval-dataset gsm8k --student-model Qwen/Qwen3-4B --student-gpu 0
!python main.py --mode eval --domain math --eval-dataset math --student-model Qwen/Qwen3-4B --student-gpu 0
!python main.py --mode eval --domain math --eval-dataset svamp --student-model Qwen/Qwen3-4B --student-gpu 0
!python main.py --mode eval --domain math --eval-dataset asdiv --student-model Qwen/Qwen3-4B --student-gpu 0
!python main.py --mode eval --domain math --eval-dataset mawps --student-model Qwen/Qwen3-4B --student-gpu 0
!python main.py --mode eval --domain logical --eval-dataset reclor --student-model Qwen/Qwen3-4B --student-gpu 0
!python main.py --mode eval --domain logical --eval-dataset anli_r2 --student-model Qwen/Qwen3-4B --student-gpu 0
!python main.py --mode eval --domain logical --eval-dataset anli_r3 --student-model Qwen/Qwen3-4B --student-gpu 0
!python main.py --mode eval --domain logical --eval-dataset bbh --student-model Qwen/Qwen3-4B --student-gpu 0
!python main.py --mode eval --domain commonsense --eval-dataset arc_c --student-model Qwen/Qwen3-4B --student-gpu 0
!python main.py --mode eval --domain commonsense --eval-dataset strategyqa --student-model Qwen/Qwen3-4B --student-gpu 0
!python main.py --mode eval --domain commonsense --eval-dataset openbookqa --student-model Qwen/Qwen3-4B --student-gpu 0

### SFT - math/gsm8k (SaFD-00/qwen3-4b-id-mas-math-gsm8k)
!python main.py --mode eval --domain math --eval-dataset gsm8k --student-model SaFD-00/qwen3-4b-id-mas-math-gsm8k --student-gpu 0
!python main.py --mode eval --domain math --eval-dataset math --student-model SaFD-00/qwen3-4b-id-mas-math-gsm8k --student-gpu 0
!python main.py --mode eval --domain math --eval-dataset svamp --student-model SaFD-00/qwen3-4b-id-mas-math-gsm8k --student-gpu 0
!python main.py --mode eval --domain math --eval-dataset asdiv --student-model SaFD-00/qwen3-4b-id-mas-math-gsm8k --student-gpu 0
!python main.py --mode eval --domain math --eval-dataset mawps --student-model SaFD-00/qwen3-4b-id-mas-math-gsm8k --student-gpu 0

### SFT - math/math (SaFD-00/qwen3-4b-id-mas-math-math)
!python main.py --mode eval --domain math --eval-dataset gsm8k --student-model SaFD-00/qwen3-4b-id-mas-math-math --student-gpu 0
!python main.py --mode eval --domain math --eval-dataset math --student-model SaFD-00/qwen3-4b-id-mas-math-math --student-gpu 0
!python main.py --mode eval --domain math --eval-dataset svamp --student-model SaFD-00/qwen3-4b-id-mas-math-math --student-gpu 0
!python main.py --mode eval --domain math --eval-dataset asdiv --student-model SaFD-00/qwen3-4b-id-mas-math-math --student-gpu 0
!python main.py --mode eval --domain math --eval-dataset mawps --student-model SaFD-00/qwen3-4b-id-mas-math-math --student-gpu 0

### SFT - logical/reclor (SaFD-00/qwen3-4b-id-mas-logical-reclor)
!python main.py --mode eval --domain logical --eval-dataset reclor --student-model SaFD-00/qwen3-4b-id-mas-logical-reclor --student-gpu 0
!python main.py --mode eval --domain logical --eval-dataset anli_r2 --student-model SaFD-00/qwen3-4b-id-mas-logical-reclor --student-gpu 0
!python main.py --mode eval --domain logical --eval-dataset anli_r3 --student-model SaFD-00/qwen3-4b-id-mas-logical-reclor --student-gpu 0
!python main.py --mode eval --domain logical --eval-dataset bbh --student-model SaFD-00/qwen3-4b-id-mas-logical-reclor --student-gpu 0

### SFT - commonsense/arc_c (SaFD-00/qwen3-4b-id-mas-commonsense-arc_c)
!python main.py --mode eval --domain commonsense --eval-dataset arc_c --student-model SaFD-00/qwen3-4b-id-mas-commonsense-arc_c --student-gpu 0
!python main.py --mode eval --domain commonsense --eval-dataset strategyqa --student-model SaFD-00/qwen3-4b-id-mas-commonsense-arc_c --student-gpu 0
!python main.py --mode eval --domain commonsense --eval-dataset openbookqa --student-model SaFD-00/qwen3-4b-id-mas-commonsense-arc_c --student-gpu 0

In [ ]:
os.chdir(IDMAS_ROOT)

## [4] Qwen3-8B Evaluation (Teacher=Qwen3-8B)

### Baseline (Qwen/Qwen3-8B)
!python main.py --mode eval --domain math --eval-dataset gsm8k --student-model Qwen/Qwen3-8B --student-gpu 0
!python main.py --mode eval --domain math --eval-dataset math --student-model Qwen/Qwen3-8B --student-gpu 0
!python main.py --mode eval --domain math --eval-dataset svamp --student-model Qwen/Qwen3-8B --student-gpu 0
!python main.py --mode eval --domain math --eval-dataset asdiv --student-model Qwen/Qwen3-8B --student-gpu 0
!python main.py --mode eval --domain math --eval-dataset mawps --student-model Qwen/Qwen3-8B --student-gpu 0
!python main.py --mode eval --domain logical --eval-dataset reclor --student-model Qwen/Qwen3-8B --student-gpu 0
!python main.py --mode eval --domain logical --eval-dataset anli_r2 --student-model Qwen/Qwen3-8B --student-gpu 0
!python main.py --mode eval --domain logical --eval-dataset anli_r3 --student-model Qwen/Qwen3-8B --student-gpu 0
!python main.py --mode eval --domain logical --eval-dataset bbh --student-model Qwen/Qwen3-8B --student-gpu 0
!python main.py --mode eval --domain commonsense --eval-dataset arc_c --student-model Qwen/Qwen3-8B --student-gpu 0
!python main.py --mode eval --domain commonsense --eval-dataset strategyqa --student-model Qwen/Qwen3-8B --student-gpu 0
!python main.py --mode eval --domain commonsense --eval-dataset openbookqa --student-model Qwen/Qwen3-8B --student-gpu 0

### SFT - math/gsm8k (SaFD-00/qwen3-8b-id-mas-math-gsm8k)
!python main.py --mode eval --domain math --eval-dataset gsm8k --student-model SaFD-00/qwen3-8b-id-mas-math-gsm8k --student-gpu 0
!python main.py --mode eval --domain math --eval-dataset math --student-model SaFD-00/qwen3-8b-id-mas-math-gsm8k --student-gpu 0
!python main.py --mode eval --domain math --eval-dataset svamp --student-model SaFD-00/qwen3-8b-id-mas-math-gsm8k --student-gpu 0
!python main.py --mode eval --domain math --eval-dataset asdiv --student-model SaFD-00/qwen3-8b-id-mas-math-gsm8k --student-gpu 0
!python main.py --mode eval --domain math --eval-dataset mawps --student-model SaFD-00/qwen3-8b-id-mas-math-gsm8k --student-gpu 0

### SFT - math/math (SaFD-00/qwen3-8b-id-mas-math-math)
!python main.py --mode eval --domain math --eval-dataset gsm8k --student-model SaFD-00/qwen3-8b-id-mas-math-math --student-gpu 0
!python main.py --mode eval --domain math --eval-dataset math --student-model SaFD-00/qwen3-8b-id-mas-math-math --student-gpu 0
!python main.py --mode eval --domain math --eval-dataset svamp --student-model SaFD-00/qwen3-8b-id-mas-math-math --student-gpu 0
!python main.py --mode eval --domain math --eval-dataset asdiv --student-model SaFD-00/qwen3-8b-id-mas-math-math --student-gpu 0
!python main.py --mode eval --domain math --eval-dataset mawps --student-model SaFD-00/qwen3-8b-id-mas-math-math --student-gpu 0

### SFT - logical/reclor (SaFD-00/qwen3-8b-id-mas-logical-reclor)
!python main.py --mode eval --domain logical --eval-dataset reclor --student-model SaFD-00/qwen3-8b-id-mas-logical-reclor --student-gpu 0
!python main.py --mode eval --domain logical --eval-dataset anli_r2 --student-model SaFD-00/qwen3-8b-id-mas-logical-reclor --student-gpu 0
!python main.py --mode eval --domain logical --eval-dataset anli_r3 --student-model SaFD-00/qwen3-8b-id-mas-logical-reclor --student-gpu 0
!python main.py --mode eval --domain logical --eval-dataset bbh --student-model SaFD-00/qwen3-8b-id-mas-logical-reclor --student-gpu 0

### SFT - commonsense/arc_c (SaFD-00/qwen3-8b-id-mas-commonsense-arc_c)
!python main.py --mode eval --domain commonsense --eval-dataset arc_c --student-model SaFD-00/qwen3-8b-id-mas-commonsense-arc_c --student-gpu 0
!python main.py --mode eval --domain commonsense --eval-dataset strategyqa --student-model SaFD-00/qwen3-8b-id-mas-commonsense-arc_c --student-gpu 0
!python main.py --mode eval --domain commonsense --eval-dataset openbookqa --student-model SaFD-00/qwen3-8b-id-mas-commonsense-arc_c --student-gpu 0

In [ ]:
os.chdir(IDMAS_ROOT)

## [5] Qwen3-14B Evaluation (Teacher=Qwen3-14B)

### Baseline (Qwen/Qwen3-14B)
!python main.py --mode eval --domain math --eval-dataset gsm8k --student-model Qwen/Qwen3-14B --student-gpu 0,1
!python main.py --mode eval --domain math --eval-dataset math --student-model Qwen/Qwen3-14B --student-gpu 0,1
!python main.py --mode eval --domain math --eval-dataset svamp --student-model Qwen/Qwen3-14B --student-gpu 0,1
!python main.py --mode eval --domain math --eval-dataset asdiv --student-model Qwen/Qwen3-14B --student-gpu 0,1
!python main.py --mode eval --domain math --eval-dataset mawps --student-model Qwen/Qwen3-14B --student-gpu 0,1
!python main.py --mode eval --domain logical --eval-dataset reclor --student-model Qwen/Qwen3-14B --student-gpu 0,1
!python main.py --mode eval --domain logical --eval-dataset anli_r2 --student-model Qwen/Qwen3-14B --student-gpu 0,1
!python main.py --mode eval --domain logical --eval-dataset anli_r3 --student-model Qwen/Qwen3-14B --student-gpu 0,1
!python main.py --mode eval --domain logical --eval-dataset bbh --student-model Qwen/Qwen3-14B --student-gpu 0,1
!python main.py --mode eval --domain commonsense --eval-dataset arc_c --student-model Qwen/Qwen3-14B --student-gpu 0,1
!python main.py --mode eval --domain commonsense --eval-dataset strategyqa --student-model Qwen/Qwen3-14B --student-gpu 0,1
!python main.py --mode eval --domain commonsense --eval-dataset openbookqa --student-model Qwen/Qwen3-14B --student-gpu 0,1

### SFT - math/gsm8k (SaFD-00/qwen3-14b-id-mas-math-gsm8k)
!python main.py --mode eval --domain math --eval-dataset gsm8k --student-model SaFD-00/qwen3-14b-id-mas-math-gsm8k --student-gpu 0,1
!python main.py --mode eval --domain math --eval-dataset math --student-model SaFD-00/qwen3-14b-id-mas-math-gsm8k --student-gpu 0,1
!python main.py --mode eval --domain math --eval-dataset svamp --student-model SaFD-00/qwen3-14b-id-mas-math-gsm8k --student-gpu 0,1
!python main.py --mode eval --domain math --eval-dataset asdiv --student-model SaFD-00/qwen3-14b-id-mas-math-gsm8k --student-gpu 0,1
!python main.py --mode eval --domain math --eval-dataset mawps --student-model SaFD-00/qwen3-14b-id-mas-math-gsm8k --student-gpu 0,1

### SFT - math/math (SaFD-00/qwen3-14b-id-mas-math-math)
!python main.py --mode eval --domain math --eval-dataset gsm8k --student-model SaFD-00/qwen3-14b-id-mas-math-math --student-gpu 0,1
!python main.py --mode eval --domain math --eval-dataset math --student-model SaFD-00/qwen3-14b-id-mas-math-math --student-gpu 0,1
!python main.py --mode eval --domain math --eval-dataset svamp --student-model SaFD-00/qwen3-14b-id-mas-math-math --student-gpu 0,1
!python main.py --mode eval --domain math --eval-dataset asdiv --student-model SaFD-00/qwen3-14b-id-mas-math-math --student-gpu 0,1
!python main.py --mode eval --domain math --eval-dataset mawps --student-model SaFD-00/qwen3-14b-id-mas-math-math --student-gpu 0,1

### SFT - logical/reclor (SaFD-00/qwen3-14b-id-mas-logical-reclor)
!python main.py --mode eval --domain logical --eval-dataset reclor --student-model SaFD-00/qwen3-14b-id-mas-logical-reclor --student-gpu 0,1
!python main.py --mode eval --domain logical --eval-dataset anli_r2 --student-model SaFD-00/qwen3-14b-id-mas-logical-reclor --student-gpu 0,1
!python main.py --mode eval --domain logical --eval-dataset anli_r3 --student-model SaFD-00/qwen3-14b-id-mas-logical-reclor --student-gpu 0,1
!python main.py --mode eval --domain logical --eval-dataset bbh --student-model SaFD-00/qwen3-14b-id-mas-logical-reclor --student-gpu 0,1

### SFT - commonsense/arc_c (SaFD-00/qwen3-14b-id-mas-commonsense-arc_c)
!python main.py --mode eval --domain commonsense --eval-dataset arc_c --student-model SaFD-00/qwen3-14b-id-mas-commonsense-arc_c --student-gpu 0,1
!python main.py --mode eval --domain commonsense --eval-dataset strategyqa --student-model SaFD-00/qwen3-14b-id-mas-commonsense-arc_c --student-gpu 0,1
!python main.py --mode eval --domain commonsense --eval-dataset openbookqa --student-model SaFD-00/qwen3-14b-id-mas-commonsense-arc_c --student-gpu 0,1

In [ ]:
os.chdir(IDMAS_ROOT)

## [6] Qwen3-32B Evaluation (Teacher=Qwen3-32B)

### Baseline (Qwen/Qwen3-32B)
!python main.py --mode eval --domain math --eval-dataset gsm8k --student-model Qwen/Qwen3-32B --student-gpu 0,1
!python main.py --mode eval --domain math --eval-dataset math --student-model Qwen/Qwen3-32B --student-gpu 0,1
!python main.py --mode eval --domain math --eval-dataset svamp --student-model Qwen/Qwen3-32B --student-gpu 0,1
!python main.py --mode eval --domain math --eval-dataset asdiv --student-model Qwen/Qwen3-32B --student-gpu 0,1
!python main.py --mode eval --domain math --eval-dataset mawps --student-model Qwen/Qwen3-32B --student-gpu 0,1
!python main.py --mode eval --domain logical --eval-dataset reclor --student-model Qwen/Qwen3-32B --student-gpu 0,1
!python main.py --mode eval --domain logical --eval-dataset anli_r2 --student-model Qwen/Qwen3-32B --student-gpu 0,1
!python main.py --mode eval --domain logical --eval-dataset anli_r3 --student-model Qwen/Qwen3-32B --student-gpu 0,1
!python main.py --mode eval --domain logical --eval-dataset bbh --student-model Qwen/Qwen3-32B --student-gpu 0,1
!python main.py --mode eval --domain commonsense --eval-dataset arc_c --student-model Qwen/Qwen3-32B --student-gpu 0,1
!python main.py --mode eval --domain commonsense --eval-dataset strategyqa --student-model Qwen/Qwen3-32B --student-gpu 0,1
!python main.py --mode eval --domain commonsense --eval-dataset openbookqa --student-model Qwen/Qwen3-32B --student-gpu 0,1

### SFT - math/gsm8k (SaFD-00/qwen3-32b-id-mas-math-gsm8k)
!python main.py --mode eval --domain math --eval-dataset gsm8k --student-model SaFD-00/qwen3-32b-id-mas-math-gsm8k --student-gpu 0,1
!python main.py --mode eval --domain math --eval-dataset math --student-model SaFD-00/qwen3-32b-id-mas-math-gsm8k --student-gpu 0,1
!python main.py --mode eval --domain math --eval-dataset svamp --student-model SaFD-00/qwen3-32b-id-mas-math-gsm8k --student-gpu 0,1
!python main.py --mode eval --domain math --eval-dataset asdiv --student-model SaFD-00/qwen3-32b-id-mas-math-gsm8k --student-gpu 0,1
!python main.py --mode eval --domain math --eval-dataset mawps --student-model SaFD-00/qwen3-32b-id-mas-math-gsm8k --student-gpu 0,1

### SFT - math/math (SaFD-00/qwen3-32b-id-mas-math-math)
!python main.py --mode eval --domain math --eval-dataset gsm8k --student-model SaFD-00/qwen3-32b-id-mas-math-math --student-gpu 0,1
!python main.py --mode eval --domain math --eval-dataset math --student-model SaFD-00/qwen3-32b-id-mas-math-math --student-gpu 0,1
!python main.py --mode eval --domain math --eval-dataset svamp --student-model SaFD-00/qwen3-32b-id-mas-math-math --student-gpu 0,1
!python main.py --mode eval --domain math --eval-dataset asdiv --student-model SaFD-00/qwen3-32b-id-mas-math-math --student-gpu 0,1
!python main.py --mode eval --domain math --eval-dataset mawps --student-model SaFD-00/qwen3-32b-id-mas-math-math --student-gpu 0,1

### SFT - logical/reclor (SaFD-00/qwen3-32b-id-mas-logical-reclor)
!python main.py --mode eval --domain logical --eval-dataset reclor --student-model SaFD-00/qwen3-32b-id-mas-logical-reclor --student-gpu 0,1
!python main.py --mode eval --domain logical --eval-dataset anli_r2 --student-model SaFD-00/qwen3-32b-id-mas-logical-reclor --student-gpu 0,1
!python main.py --mode eval --domain logical --eval-dataset anli_r3 --student-model SaFD-00/qwen3-32b-id-mas-logical-reclor --student-gpu 0,1
!python main.py --mode eval --domain logical --eval-dataset bbh --student-model SaFD-00/qwen3-32b-id-mas-logical-reclor --student-gpu 0,1

### SFT - commonsense/arc_c (SaFD-00/qwen3-32b-id-mas-commonsense-arc_c)
!python main.py --mode eval --domain commonsense --eval-dataset arc_c --student-model SaFD-00/qwen3-32b-id-mas-commonsense-arc_c --student-gpu 0,1
!python main.py --mode eval --domain commonsense --eval-dataset strategyqa --student-model SaFD-00/qwen3-32b-id-mas-commonsense-arc_c --student-gpu 0,1
!python main.py --mode eval --domain commonsense --eval-dataset openbookqa --student-model SaFD-00/qwen3-32b-id-mas-commonsense-arc_c --student-gpu 0,1

### 3.2 Cross-Model Distillation Evaluation

실험 [7]~[8]: GPT-5.2 Teacher로 학습한 모델의 성능 평가

In [ ]:
os.chdir(IDMAS_ROOT)

## [7] Qwen3-4B Evaluation (Teacher=gpt-5.2)

### Baseline (Qwen/Qwen3-4B)
!python main.py --mode eval --domain math --eval-dataset gsm8k --student-model Qwen/Qwen3-4B --student-gpu 0
!python main.py --mode eval --domain math --eval-dataset math --student-model Qwen/Qwen3-4B --student-gpu 0
!python main.py --mode eval --domain math --eval-dataset svamp --student-model Qwen/Qwen3-4B --student-gpu 0
!python main.py --mode eval --domain math --eval-dataset asdiv --student-model Qwen/Qwen3-4B --student-gpu 0
!python main.py --mode eval --domain math --eval-dataset mawps --student-model Qwen/Qwen3-4B --student-gpu 0
!python main.py --mode eval --domain logical --eval-dataset reclor --student-model Qwen/Qwen3-4B --student-gpu 0
!python main.py --mode eval --domain logical --eval-dataset anli_r2 --student-model Qwen/Qwen3-4B --student-gpu 0
!python main.py --mode eval --domain logical --eval-dataset anli_r3 --student-model Qwen/Qwen3-4B --student-gpu 0
!python main.py --mode eval --domain logical --eval-dataset bbh --student-model Qwen/Qwen3-4B --student-gpu 0
!python main.py --mode eval --domain commonsense --eval-dataset arc_c --student-model Qwen/Qwen3-4B --student-gpu 0
!python main.py --mode eval --domain commonsense --eval-dataset strategyqa --student-model Qwen/Qwen3-4B --student-gpu 0
!python main.py --mode eval --domain commonsense --eval-dataset openbookqa --student-model Qwen/Qwen3-4B --student-gpu 0

### SFT - math/gsm8k (SaFD-00/qwen3-4b-id-mas-math-gsm8k-gpt52)
!python main.py --mode eval --domain math --eval-dataset gsm8k --student-model SaFD-00/qwen3-4b-id-mas-math-gsm8k-gpt52 --student-gpu 0
!python main.py --mode eval --domain math --eval-dataset math --student-model SaFD-00/qwen3-4b-id-mas-math-gsm8k-gpt52 --student-gpu 0
!python main.py --mode eval --domain math --eval-dataset svamp --student-model SaFD-00/qwen3-4b-id-mas-math-gsm8k-gpt52 --student-gpu 0
!python main.py --mode eval --domain math --eval-dataset asdiv --student-model SaFD-00/qwen3-4b-id-mas-math-gsm8k-gpt52 --student-gpu 0
!python main.py --mode eval --domain math --eval-dataset mawps --student-model SaFD-00/qwen3-4b-id-mas-math-gsm8k-gpt52 --student-gpu 0

### SFT - math/math (SaFD-00/qwen3-4b-id-mas-math-math-gpt52)
!python main.py --mode eval --domain math --eval-dataset gsm8k --student-model SaFD-00/qwen3-4b-id-mas-math-math-gpt52 --student-gpu 0
!python main.py --mode eval --domain math --eval-dataset math --student-model SaFD-00/qwen3-4b-id-mas-math-math-gpt52 --student-gpu 0
!python main.py --mode eval --domain math --eval-dataset svamp --student-model SaFD-00/qwen3-4b-id-mas-math-math-gpt52 --student-gpu 0
!python main.py --mode eval --domain math --eval-dataset asdiv --student-model SaFD-00/qwen3-4b-id-mas-math-math-gpt52 --student-gpu 0
!python main.py --mode eval --domain math --eval-dataset mawps --student-model SaFD-00/qwen3-4b-id-mas-math-math-gpt52 --student-gpu 0

### SFT - logical/reclor (SaFD-00/qwen3-4b-id-mas-logical-reclor-gpt52)
!python main.py --mode eval --domain logical --eval-dataset reclor --student-model SaFD-00/qwen3-4b-id-mas-logical-reclor-gpt52 --student-gpu 0
!python main.py --mode eval --domain logical --eval-dataset anli_r2 --student-model SaFD-00/qwen3-4b-id-mas-logical-reclor-gpt52 --student-gpu 0
!python main.py --mode eval --domain logical --eval-dataset anli_r3 --student-model SaFD-00/qwen3-4b-id-mas-logical-reclor-gpt52 --student-gpu 0
!python main.py --mode eval --domain logical --eval-dataset bbh --student-model SaFD-00/qwen3-4b-id-mas-logical-reclor-gpt52 --student-gpu 0

### SFT - commonsense/arc_c (SaFD-00/qwen3-4b-id-mas-commonsense-arc_c-gpt52)
!python main.py --mode eval --domain commonsense --eval-dataset arc_c --student-model SaFD-00/qwen3-4b-id-mas-commonsense-arc_c-gpt52 --student-gpu 0
!python main.py --mode eval --domain commonsense --eval-dataset strategyqa --student-model SaFD-00/qwen3-4b-id-mas-commonsense-arc_c-gpt52 --student-gpu 0
!python main.py --mode eval --domain commonsense --eval-dataset openbookqa --student-model SaFD-00/qwen3-4b-id-mas-commonsense-arc_c-gpt52 --student-gpu 0

In [ ]:
os.chdir(IDMAS_ROOT)

## [8] Qwen3-8B Evaluation (Teacher=gpt-5.2)

### Baseline (Qwen/Qwen3-8B)
!python main.py --mode eval --domain math --eval-dataset gsm8k --student-model Qwen/Qwen3-8B --student-gpu 0
!python main.py --mode eval --domain math --eval-dataset math --student-model Qwen/Qwen3-8B --student-gpu 0
!python main.py --mode eval --domain math --eval-dataset svamp --student-model Qwen/Qwen3-8B --student-gpu 0
!python main.py --mode eval --domain math --eval-dataset asdiv --student-model Qwen/Qwen3-8B --student-gpu 0
!python main.py --mode eval --domain math --eval-dataset mawps --student-model Qwen/Qwen3-8B --student-gpu 0
!python main.py --mode eval --domain logical --eval-dataset reclor --student-model Qwen/Qwen3-8B --student-gpu 0
!python main.py --mode eval --domain logical --eval-dataset anli_r2 --student-model Qwen/Qwen3-8B --student-gpu 0
!python main.py --mode eval --domain logical --eval-dataset anli_r3 --student-model Qwen/Qwen3-8B --student-gpu 0
!python main.py --mode eval --domain logical --eval-dataset bbh --student-model Qwen/Qwen3-8B --student-gpu 0
!python main.py --mode eval --domain commonsense --eval-dataset arc_c --student-model Qwen/Qwen3-8B --student-gpu 0
!python main.py --mode eval --domain commonsense --eval-dataset strategyqa --student-model Qwen/Qwen3-8B --student-gpu 0
!python main.py --mode eval --domain commonsense --eval-dataset openbookqa --student-model Qwen/Qwen3-8B --student-gpu 0

### SFT - math/gsm8k (SaFD-00/qwen3-8b-id-mas-math-gsm8k-gpt52)
!python main.py --mode eval --domain math --eval-dataset gsm8k --student-model SaFD-00/qwen3-8b-id-mas-math-gsm8k-gpt52 --student-gpu 0
!python main.py --mode eval --domain math --eval-dataset math --student-model SaFD-00/qwen3-8b-id-mas-math-gsm8k-gpt52 --student-gpu 0
!python main.py --mode eval --domain math --eval-dataset svamp --student-model SaFD-00/qwen3-8b-id-mas-math-gsm8k-gpt52 --student-gpu 0
!python main.py --mode eval --domain math --eval-dataset asdiv --student-model SaFD-00/qwen3-8b-id-mas-math-gsm8k-gpt52 --student-gpu 0
!python main.py --mode eval --domain math --eval-dataset mawps --student-model SaFD-00/qwen3-8b-id-mas-math-gsm8k-gpt52 --student-gpu 0

### SFT - math/math (SaFD-00/qwen3-8b-id-mas-math-math-gpt52)
!python main.py --mode eval --domain math --eval-dataset gsm8k --student-model SaFD-00/qwen3-8b-id-mas-math-math-gpt52 --student-gpu 0
!python main.py --mode eval --domain math --eval-dataset math --student-model SaFD-00/qwen3-8b-id-mas-math-math-gpt52 --student-gpu 0
!python main.py --mode eval --domain math --eval-dataset svamp --student-model SaFD-00/qwen3-8b-id-mas-math-math-gpt52 --student-gpu 0
!python main.py --mode eval --domain math --eval-dataset asdiv --student-model SaFD-00/qwen3-8b-id-mas-math-math-gpt52 --student-gpu 0
!python main.py --mode eval --domain math --eval-dataset mawps --student-model SaFD-00/qwen3-8b-id-mas-math-math-gpt52 --student-gpu 0

### SFT - logical/reclor (SaFD-00/qwen3-8b-id-mas-logical-reclor-gpt52)
!python main.py --mode eval --domain logical --eval-dataset reclor --student-model SaFD-00/qwen3-8b-id-mas-logical-reclor-gpt52 --student-gpu 0
!python main.py --mode eval --domain logical --eval-dataset anli_r2 --student-model SaFD-00/qwen3-8b-id-mas-logical-reclor-gpt52 --student-gpu 0
!python main.py --mode eval --domain logical --eval-dataset anli_r3 --student-model SaFD-00/qwen3-8b-id-mas-logical-reclor-gpt52 --student-gpu 0
!python main.py --mode eval --domain logical --eval-dataset bbh --student-model SaFD-00/qwen3-8b-id-mas-logical-reclor-gpt52 --student-gpu 0

### SFT - commonsense/arc_c (SaFD-00/qwen3-8b-id-mas-commonsense-arc_c-gpt52)
!python main.py --mode eval --domain commonsense --eval-dataset arc_c --student-model SaFD-00/qwen3-8b-id-mas-commonsense-arc_c-gpt52 --student-gpu 0
!python main.py --mode eval --domain commonsense --eval-dataset strategyqa --student-model SaFD-00/qwen3-8b-id-mas-commonsense-arc_c-gpt52 --student-gpu 0
!python main.py --mode eval --domain commonsense --eval-dataset openbookqa --student-model SaFD-00/qwen3-8b-id-mas-commonsense-arc_c-gpt52 --student-gpu 0